# PS3: Address Geocoder (Demo/Lightweight Pipeline)

This notebook builds a simple geocoder that learns from successful field visits.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"

In [ ]:
addresses = pd.read_csv(os.path.join(base_path, "shared", "addresses.csv"))
baseline = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "baseline_geocodes.csv"))

In [ ]:
field_visits = pd.read_csv(os.path.join(base_path, "shared", "field_visits.csv"))
surveyed = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "surveyed_addresses.csv"))

## 3. Data Processing

In [ ]:
success_outcomes = ['cash_collected', 'ptp_collected', 'met_borrower']
successful_visits = field_visits[field_visits['outcome'].isin(success_outcomes)]

In [ ]:
address_gps = successful_visits.groupby('address_id').agg(
    visit_x=('checkin_x', 'mean'),
    visit_y=('checkin_y', 'mean')
).reset_index()

In [ ]:
data = pd.merge(addresses, baseline, on='address_id', how='left')
data = pd.merge(data, address_gps, on='address_id', how='left')

In [ ]:
data['pred_x'] = data['visit_x'].fillna(data['geocoder_x'])
data['pred_y'] = data['visit_y'].fillna(data['geocoder_y'])

## 4. Evaluation

In [ ]:
eval_data = pd.merge(surveyed, data, on='address_id', how='inner')

In [ ]:
eval_data['model_error'] = np.sqrt((eval_data['pred_x'] - eval_data['surveyed_x'])**2 + (eval_data['pred_y'] - eval_data['surveyed_y'])**2)

In [ ]:
print("Lightweight Model Metrics (m):")
print(f"Median Error: {eval_data['model_error'].median():.2f}")
print(f"90th Percentile: {eval_data['model_error'].quantile(0.90):.2f}")